# Maya 2 voice server on a free Kaggle GPU
Same server as the Colab notebook (English, Hindi, Hinglish; first sound in about 0.13 s), for when Colab is not available. Also used to measure latency: run `npx tsx scripts/latency-maya2-avatar.mts` on your Mac after step 4.

1. Open kaggle.com → Create → New Notebook → File → Import Notebook, and choose this file.
2. In the right panel (Session options): **Accelerator → GPU T4 x2**, **Internet → On** (Internet needs a phone-verified Kaggle account: Settings → Phone verification).
3. Choose **Run All**. Setup takes about 5 minutes.
4. The last cell prints `MAYA2_URL=...` and `MAYA2_TOKEN=...`. Copy both lines, then on your Mac run `bash scripts/set-links.sh`.

Keep the notebook session open (Kaggle stops it after about 20 minutes idle). The link changes every run.

In [ ]:
!nvidia-smi | head -15
!ffmpeg -version | head -1


In [ ]:
!pip -q install "maya3[server] @ git+https://github.com/MayaResearch/maya2-native" aiohttp
# Installing maya3 moves torch to 2.8.0, which breaks the torchvision Colab ships with (the server then dies with "operator torchvision::nms does not exist"). Put a matching pair back.
!pip -q install "torchvision==0.23.0" "torchaudio==2.8.0" --index-url https://download.pytorch.org/whl/cu126
import torch, torchvision; print(torch.__version__, torchvision.__version__)

In [ ]:
import subprocess, time, urllib.request
subprocess.run(['pkill', '-f', 'maya3.server']); time.sleep(3)  # an older copy would hold port 8090
server = subprocess.Popen(['python', '-m', 'maya3.server', '--ckpt', 'maya-research/maya2-native', '--host', '127.0.0.1', '--port', '8090', '--max-seats', '2'], stdout=open('/kaggle/working/maya.log', 'w'), stderr=subprocess.STDOUT)
for i in range(180):
    try:
        if urllib.request.urlopen('http://127.0.0.1:8090/health', timeout=2).status == 200: print('Maya 2 ready'); break
    except Exception: time.sleep(5)
else:
    print(open('/kaggle/working/maya.log').read()[-3000:])
print(open('/kaggle/working/maya.log').read()[-1500:])

The Maya 2 server has no login of its own, so this small gate only lets requests with the secret token through. It also adds speed control (Maya 2 itself only does 1.0): the app sends a `speed` of 0.6 to 1.6 and the gate speeds the audio up cleanly with ffmpeg while it streams.

In [ ]:
import secrets, asyncio, threading, json
from aiohttp import web, ClientSession
TOKEN = secrets.token_urlsafe(32)
# Maya 2's own server refuses any speed but 1.0, and stretching finished audio in the app blurs it. So the gate takes a "speed" field
# (0.6 to 1.6) and speeds the audio up here with ffmpeg's atempo, as it streams (pcm) or on the whole file (wav): no extra wait for the first sound.
async def tempo_pcm(r, resp, speed):
    p = await asyncio.create_subprocess_exec('ffmpeg', '-loglevel', 'error', '-f', 's16le', '-ar', '24000', '-ac', '1', '-i', 'pipe:0', '-af', f'atempo={speed:.3f}', '-f', 's16le', 'pipe:1',
                                             stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE)
    async def feed():
        try:
            async for c in r.content.iter_any():
                p.stdin.write(c); await p.stdin.drain()
        finally:
            p.stdin.close()
    t = asyncio.create_task(feed())
    while True:
        d = await p.stdout.read(4800)
        if not d: break
        await resp.write(d)
    await t; await p.wait()
async def gate(request):
    if request.headers.get('authorization') != f'Bearer {TOKEN}':
        return web.Response(status=401, text='bad token')
    body = await request.read()
    speech = request.method == 'POST' and request.path.startswith('/v1/audio/speech')
    speed, fmt = 1.0, 'wav'
    if speech:
        try:
            j = json.loads(body); speed = float(j.pop('speed', 1.0) or 1.0); fmt = j.get('response_format', 'wav'); body = json.dumps(j).encode()
        except Exception: pass
    speed = min(1.6, max(0.6, speed))
    fast = speech and abs(speed - 1.0) > 0.01
    async with ClientSession() as s:
        async with s.request(request.method, f'http://127.0.0.1:8090{request.path_qs}', data=body or None,
                             headers={k: v for k, v in request.headers.items() if k.lower() in ('content-type',)}) as r:
            hdrs = {k: v for k, v in r.headers.items() if k.lower() in ('content-type', 'x-sample-rate', 'x-encoding')}
            if fast and r.status == 200 and fmt == 'wav':
                raw = await r.read()
                p = await asyncio.create_subprocess_exec('ffmpeg', '-loglevel', 'error', '-i', 'pipe:0', '-af', f'atempo={speed:.3f}', '-f', 'wav', 'pipe:1', stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE)
                out, _ = await p.communicate(raw)
                return web.Response(status=200, body=out, headers=hdrs)
            resp = web.StreamResponse(status=r.status, headers=hdrs)
            await resp.prepare(request)
            if fast and r.status == 200 and fmt == 'pcm':
                await tempo_pcm(r, resp, speed)
            else:
                async for chunk in r.content.iter_any(): await resp.write(chunk)
            await resp.write_eof(); return resp
async def start():
    app = web.Application(client_max_size=64 * 1024 * 1024); app.router.add_route('*', '/{tail:.*}', gate)
    runner = web.AppRunner(app); await runner.setup(); await web.TCPSite(runner, '127.0.0.1', 8081).start()
def run():
    loop = asyncio.new_event_loop(); loop.run_until_complete(start()); loop.run_forever()
threading.Thread(target=run, daemon=True).start(); time.sleep(2); print('gate on 8081')
import urllib.request as _u
_r = _u.Request('http://127.0.0.1:8081/v1/voices', headers={'authorization': f'Bearer {TOKEN}'})
print('self-check through the gate:', _u.urlopen(_r, timeout=10).status, '(200 means Maya 2 is answering)')

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared && chmod +x /kaggle/working/cloudflared
import re
tunnel = subprocess.Popen(['/kaggle/working/cloudflared', 'tunnel', '--protocol', 'http2', '--url', 'http://127.0.0.1:8081'], stdout=open('/kaggle/working/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for i in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/kaggle/working/tunnel.log').read())
    if m: url = m.group(0); break
    time.sleep(2)
print()
print('Paste these two lines into apps/web/.env.local AND apps/voice/.env.local:')
print()
print(f'MAYA2_URL={url}')
print(f'MAYA2_TOKEN={TOKEN}')

In [ ]:
print(f'MAYA2_URL={url}')
print(f'MAYA2_TOKEN={TOKEN}')
